# Stage 0 — Assembly & Event Clock

Ingest desk intraday tape + Bloomberg CSV exports, clean, validate, build event clock.

**Outputs saved to disk:**
- `data/cache/intraday_raw.parquet` — cleaned intraday tape
- `intraday_df`, `bloomberg_df`, `daily_df` — passed to Stage 1

Ref: Fleming, Hrung, Mizrach & Romero (intraday auction effects).

In [ ]:
import sys, os
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from config import DATA_DIR, CACHE_DIR, INTRADAY_CSV_PATH, BLOOMBERG_CSV_PATH
from src.assembly import (
    upload_and_inspect,
    clean_intraday_csv,
    clean_bloomberg_csv,
    validate_uploads,
    load_intraday,
    load_bloomberg,
    add_event_clock,
    build_daily_curve,
)

print('Data dir:', DATA_DIR)
print('Cache dir:', CACHE_DIR)

## 0a. Inspect raw files

Drop your CSVs into `data/` then run the cell below.
Fill in `INTRADAY_COL_MAP` and `BLOOMBERG_COL_MAP` in `config.py` based on the column names you see.

In [ ]:
# Inspect intraday tape
raw_intra = upload_and_inspect(INTRADAY_CSV_PATH, skip_rows=0)

In [ ]:
# Inspect Bloomberg auction results
raw_bb = upload_and_inspect(BLOOMBERG_CSV_PATH, skip_rows=0)

## 0b. Load & clean

In [ ]:
intraday_df  = load_intraday(use_cache=False)   # set use_cache=True after first run
bloomberg_df = load_bloomberg()

print(f'\nIntraday : {len(intraday_df):,} rows  |  {intraday_df["auction_id"].nunique()} auctions')
print(f'Bloomberg: {len(bloomberg_df):,} auctions')

## 0c. Validate schemas

In [ ]:
ok = validate_uploads(intraday_df, bloomberg_df)
assert ok, 'Fix schema issues above before proceeding.'

## 0d. Build event clock

In [ ]:
intraday_df = add_event_clock(intraday_df)
intraday_df[['auction_id', 'timestamp_et', 'event_minute', 'phase', 'otr_30y_yield']].head(10)

## 0e. Build daily curve table

In [ ]:
daily_df = build_daily_curve(bloomberg_df, intraday_df)
print(f'Daily curve: {len(daily_df)} rows  cols={list(daily_df.columns[:8])} …')
daily_df.head()

## 0f. Persist for downstream notebooks

In [ ]:
intraday_df.to_parquet(CACHE_DIR / 'intraday_evtclk.parquet', index=False)
bloomberg_df.to_parquet(CACHE_DIR / 'bloomberg.parquet', index=False)
daily_df.to_parquet(CACHE_DIR / 'daily_curve.parquet')
print('Saved to cache/')